In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'

In [ ]:
from utils_SI10F_scomat import *
import numpy as np
import torch
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt
import ot
from tqdm import tqdm
import matplotlib.pyplot as plt
import math
import pandas as pd

In [ ]:
device = torch.device("cpu")

In [ ]:
beta_true = torch.tensor([0.05, 0.02, 0.04, 0.06, 0.08, 0.1, 0.12, 0.14, 0.16, 0.18, 0.2, 0.05])

In [ ]:
def normalized_weight(log_w):
    tmp_w = (log_w - log_w.max()).exp()
    tmp_w = tmp_w / tmp_w.sum()

    thresh_id = 0
    while tmp_w.max() > 100 / log_w.shape[0]:
        thresh_id += 1
        clip = log_w.sort(descending=True).values[thresh_id]
        log_w = torch.clamp(log_w, max = clip)
        tmp_w = (log_w - log_w.max()).exp()
        tmp_w = tmp_w / tmp_w.sum()
    return tmp_w

def weighted_quantile(array, weight, quantile):
    # weight sums to 1
    sorter = np.argsort(array)
    sorted_array = array[sorter]
    sorted_weight = weight[sorter]
    index = np.searchsorted(np.cumsum(sorted_weight), quantile, side='left')
    return sorted_array[index]

### Our method

In [ ]:
abs_error = [] # |post_mean - truth|, element-wise 
CI90_width = []
Cover90 = []
CI95_width = []
Cover95 = []

for obs_id in range(50):
    post_samples = torch.tensor(np.load(f"Langevin_res/post_samples_obs{obs_id}.npy"), dtype=torch.float32)
   
    abs_error.append( (post_samples.mean(dim = 0) - beta_true.cpu()).abs() ) 
    CI90_width.append( torch.quantile(post_samples, 0.95, dim=0) - torch.quantile(post_samples, 0.05, dim=0) )
    Cover90.append( torch.logical_and( beta_true.cpu() > torch.quantile(post_samples, 0.05, dim=0), beta_true.cpu() < torch.quantile(post_samples, 0.95, dim=0) ) )
    CI95_width.append( torch.quantile(post_samples, 0.975, dim=0) - torch.quantile(post_samples, 0.025, dim=0) )
    Cover95.append( torch.logical_and( beta_true.cpu() > torch.quantile(post_samples, 0.025, dim=0), beta_true.cpu() < torch.quantile(post_samples, 0.975, dim=0) ) )

In [ ]:
abs_error, CI90_width, Cover90, CI95_width, Cover95 = torch.stack(abs_error).numpy(), torch.stack(CI90_width).numpy(), torch.stack(Cover90).numpy(), torch.stack(CI95_width).numpy(), torch.stack(Cover95).numpy()

In [ ]:
# Calculate mean and std
mean_values = {
    "$\\lvert \\theta - \\theta_* \\rvert$": abs_error.mean(axis=0),
    "CI90_width": CI90_width.mean(axis=0),
    "Cover90": Cover90.mean(axis=0),
    "CI95_width": CI95_width.mean(axis=0),
    "Cover95": Cover95.mean(axis=0),
}

std_values = {
    "$\\lvert \\theta - \\theta_* \\rvert$": abs_error.std(axis=0),
    "CI90_width": CI90_width.std(axis=0),
    "CI95_width": CI95_width.std(axis=0),
}

# Construct the dataframe
df = pd.DataFrame({
    key: [
        f"{mean:.3f} ({std:.3f})" if key in std_values else f"{mean}"
        for mean, std in zip(mean_values[key], std_values.get(key, [None] * len(mean_values[key])))
    ]
    for key in mean_values
})

# Set the index
df.index = ["Facility"] + [f"Floor {i+1}" for i in range(10)] + ["Room"]

In [ ]:
df.insert(0, '$\theta_*$', beta_true)

In [ ]:
df

In [ ]:
df.to_csv("Our_method.csv", index=False)

### BSL

In [ ]:
abs_error_BSL = [] # |post_mean - truth|, element-wise 
CI90_width_BSL = []
Cover90_BSL = []
CI95_width_BSL = []
Cover95_BSL = []

beta_true = torch.tensor([0.05, 0.02, 0.04, 0.06, 0.08, 0.1, 0.12, 0.14, 0.16, 0.18, 0.2, 0.05]).to(device)
for task_id in range(50):
    # BSL
    log_theta_r1 = torch.tensor(np.load(f"BSL_res/BSL_task_{task_id}.npy"), dtype=torch.float32)
    theta_r1 = log_theta_r1.exp()
    
    abs_error_BSL.append( (theta_r1.mean(dim = 0) - beta_true.cpu()).abs() ) 
    CI90_width_BSL.append( torch.quantile(theta_r1, 0.95, dim=0) - torch.quantile(theta_r1, 0.05, dim=0) )
    Cover90_BSL.append( torch.logical_and( beta_true.cpu() > torch.quantile(theta_r1, 0.05, dim=0), beta_true.cpu() < torch.quantile(theta_r1, 0.95, dim=0) ) )
    CI95_width_BSL.append( torch.quantile(theta_r1, 0.975, dim=0) - torch.quantile(theta_r1, 0.025, dim=0) )
    Cover95_BSL.append( torch.logical_and( beta_true.cpu() > torch.quantile(theta_r1, 0.025, dim=0), beta_true.cpu() < torch.quantile(theta_r1, 0.975, dim=0) ) )

In [ ]:
abs_error_BSL, CI90_width_BSL, Cover90_BSL, CI95_width_BSL, Cover95_BSL = torch.stack(abs_error_BSL).numpy(), torch.stack(CI90_width_BSL).numpy(), torch.stack(Cover90_BSL).numpy(), torch.stack(CI95_width_BSL).numpy(), torch.stack(Cover95_BSL).numpy()

In [ ]:
# Calculate mean and std
mean_values = {
    "$\\lvert \\theta - \\theta_* \\rvert$": abs_error_BSL.mean(axis=0),
    "CI90_width": CI90_width_BSL.mean(axis=0),
    "Cover90": Cover90_BSL.mean(axis=0),
    "CI95_width": CI95_width_BSL.mean(axis=0),
    "Cover95": Cover95_BSL.mean(axis=0),
}

std_values = {
    "$\\lvert \\theta - \\theta_* \\rvert$": abs_error_BSL.std(axis=0),
    "CI90_width": CI90_width_BSL.std(axis=0),
    "CI95_width": CI95_width_BSL.std(axis=0),
}

# Construct the dataframe
df_BSL = pd.DataFrame({
    key: [
        f"{mean:.3f} ({std:.3f})" if key in std_values else f"{mean}"
        for mean, std in zip(mean_values[key], std_values.get(key, [None] * len(mean_values[key])))
    ]
    for key in mean_values
})

# Set the index
df_BSL.index = ["Facility"] + [f"Floor {i+1}" for i in range(10)] + ["Room"]

In [ ]:
df_BSL.insert(0, '$\theta_*$', beta_true.cpu())

In [ ]:
df_BSL

In [ ]:
df_BSL.to_csv("BSL.csv", index=False)

### ABC

In [ ]:
abs_error_ABCW1 = [] # |post_mean - truth|, element-wise 
CI90_width_ABCW1 = []
Cover90_ABCW1 = []
CI95_width_ABCW1 = []
Cover95_ABCW1 = []

beta_true = torch.tensor([0.05, 0.02, 0.04, 0.06, 0.08, 0.1, 0.12, 0.14, 0.16, 0.18, 0.2, 0.05]).to(device)
for task_id in range(50):
    # ABC_W1
    log_theta_r1 = pd.read_csv(f"ABC_res/log_theta_r1_ABCW1_task{task_id}.csv")
    log_theta_r1 = torch.tensor(log_theta_r1.values, dtype=torch.float32).contiguous()
    theta_r1 = log_theta_r1.exp()

    precond_samples = pd.read_csv(f"res_precond/pre_samples_lam0_task{task_id}.csv")
    precond_samples = torch.tensor(precond_samples.values, dtype=torch.float32).contiguous()

    # calculate weight: log_weight = log \pi(\theta) - log q(\theta)
    mean_theta = precond_samples.mean(dim = 0)
    std_theta = precond_samples.std(dim = 0)
    dist_prop = torch.distributions.multivariate_normal.MultivariateNormal(loc = mean_theta, covariance_matrix = (std_theta**2).diag())
    dist_prior = torch.distributions.multivariate_normal.MultivariateNormal(loc = -3 * torch.ones(12), covariance_matrix = 4 * torch.eye(12))
    log_ABCW1_weight = dist_prior.log_prob(log_theta_r1) - dist_prop.log_prob(log_theta_r1) 
    ABCW1_weight = normalized_weight(log_ABCW1_weight)

    # weighted mean
    theta_r1 = log_theta_r1.exp()
    mean_ABCW1 = ( theta_r1 * ABCW1_weight.view(-1, 1).repeat(1, 12) ).sum(dim = 0)
    # weighted quantile
    ABCW1_qt025 = torch.zeros(12)
    ABCW1_qt975 = torch.zeros(12)
    ABCW1_qt05 = torch.zeros(12)
    ABCW1_qt95 = torch.zeros(12)
    for j in range(12):
        ABCW1_qt025[j] = weighted_quantile(theta_r1[:, j], ABCW1_weight, 0.025)
        ABCW1_qt975[j] = weighted_quantile(theta_r1[:, j], ABCW1_weight, 0.975)
        ABCW1_qt05[j] = weighted_quantile(theta_r1[:, j], ABCW1_weight, 0.05)
        ABCW1_qt95[j] = weighted_quantile(theta_r1[:, j], ABCW1_weight, 0.95)
    
    abs_error_ABCW1.append( (mean_ABCW1 - beta_true.cpu()).abs() ) 
    CI90_width_ABCW1.append( ABCW1_qt95 - ABCW1_qt05 )
    Cover90_ABCW1.append( torch.logical_and( beta_true.cpu() > ABCW1_qt05, beta_true.cpu() < ABCW1_qt95 ) )
    CI95_width_ABCW1.append( ABCW1_qt975 - ABCW1_qt025 )
    Cover95_ABCW1.append( torch.logical_and( beta_true.cpu() > ABCW1_qt025, beta_true.cpu() < ABCW1_qt975 ) )


In [ ]:
abs_error_ABCW1, CI90_width_ABCW1, Cover90_ABCW1, CI95_width_ABCW1, Cover95_ABCW1 = torch.stack(abs_error_ABCW1).numpy(), torch.stack(CI90_width_ABCW1).numpy(), torch.stack(Cover90_ABCW1).numpy(), torch.stack(CI95_width_ABCW1).numpy(), torch.stack(Cover95_ABCW1).numpy()

In [ ]:
# Calculate mean and std
mean_values = {
    "$\\lvert \\theta - \\theta_* \\rvert$": abs_error_ABCW1.mean(axis=0),
    "CI90_width": CI90_width_ABCW1.mean(axis=0),
    "Cover90": Cover90_ABCW1.mean(axis=0),
    "CI95_width": CI95_width_ABCW1.mean(axis=0),
    "Cover95": Cover95_ABCW1.mean(axis=0),
}

std_values = {
    "$\\lvert \\theta - \\theta_* \\rvert$": abs_error_ABCW1.std(axis=0),
    "CI90_width": CI90_width_ABCW1.std(axis=0),
    "CI95_width": CI95_width_ABCW1.std(axis=0),
}

# Construct the dataframe
df_ABCW1 = pd.DataFrame({
    key: [
        f"{mean:.3f} ({std:.3f})" if key in std_values else f"{mean}"
        for mean, std in zip(mean_values[key], std_values.get(key, [None] * len(mean_values[key])))
    ]
    for key in mean_values
})

# Set the index
df_ABCW1.index = ["Facility"] + [f"Floor {i+1}" for i in range(10)] + ["Room"]

In [ ]:
df_ABCW1.insert(0, '$\theta_*$', beta_true)

In [ ]:
df_ABCW1

In [ ]:
df_ABCW1.to_csv("ABCW1.csv", index=False)

### NPE

In [ ]:
abs_error_NPE = [] # |post_mean - truth|, element-wise 
CI90_width_NPE = []
Cover90_NPE = []
CI95_width_NPE = []
Cover95_NPE = []

beta_true = torch.tensor([0.05, 0.02, 0.04, 0.06, 0.08, 0.1, 0.12, 0.14, 0.16, 0.18, 0.2, 0.05]).to(device)
for task_id in range(50):
    precond_samples = pd.read_csv(f"res_precond/pre_samples_lam0_task{task_id}.csv")
    precond_samples = torch.tensor(precond_samples.values, dtype=torch.float32).contiguous()
   
    # NPE
    mu = torch.tensor(np.load(f"NPE_res/mu_obs{task_id}.npy"), dtype=torch.float32)
    Cov = torch.tensor(np.load(f"NPE_res/Cov_obs{task_id}.npy"), dtype=torch.float32)

    log_theta_r1 = torch.distributions.multivariate_normal.MultivariateNormal(
        loc = mu, covariance_matrix = Cov ).sample((10000, ))
    
    # calculate weight: log_weight = log \pi(\theta) - log q(\theta)
    mean_theta = precond_samples.mean(dim = 0)
    std_theta = precond_samples.std(dim = 0)
    dist_prop = torch.distributions.multivariate_normal.MultivariateNormal(loc = mean_theta, covariance_matrix = (std_theta**2).diag())
    dist_prior = torch.distributions.multivariate_normal.MultivariateNormal(loc = -3 * torch.ones(12), covariance_matrix = 4 * torch.eye(12))
    log_NPE_weight = dist_prior.log_prob(log_theta_r1) - dist_prop.log_prob(log_theta_r1) 
    NPE_weight = normalized_weight(log_NPE_weight)

    # weighted mean
    theta_r1 = log_theta_r1.exp()
    mean_NPE = ( theta_r1 * NPE_weight.view(-1, 1).repeat(1, 12) ).sum(dim = 0)
    # weighted quantile
    NPE_qt025 = torch.zeros(12)
    NPE_qt975 = torch.zeros(12)
    NPE_qt05 = torch.zeros(12)
    NPE_qt95 = torch.zeros(12)
    for j in range(12):
        NPE_qt025[j] = weighted_quantile(theta_r1[:, j], NPE_weight, 0.025)
        NPE_qt975[j] = weighted_quantile(theta_r1[:, j], NPE_weight, 0.975)
        NPE_qt05[j] = weighted_quantile(theta_r1[:, j], NPE_weight, 0.05)
        NPE_qt95[j] = weighted_quantile(theta_r1[:, j], NPE_weight, 0.95)
    
    abs_error_NPE.append( (mean_NPE - beta_true.cpu()).abs() ) 
    CI90_width_NPE.append( NPE_qt95 - NPE_qt05 )
    Cover90_NPE.append( torch.logical_and( beta_true.cpu() > NPE_qt05, beta_true.cpu() < NPE_qt95 ) )
    CI95_width_NPE.append( NPE_qt975 - NPE_qt025 )
    Cover95_NPE.append( torch.logical_and( beta_true.cpu() > NPE_qt025, beta_true.cpu() < NPE_qt975 ) )

In [ ]:
abs_error_NPE, CI90_width_NPE, Cover90_NPE, CI95_width_NPE, Cover95_NPE = torch.stack(abs_error_NPE).numpy(), torch.stack(CI90_width_NPE).numpy(), torch.stack(Cover90_NPE).numpy(), torch.stack(CI95_width_NPE).numpy(), torch.stack(Cover95_NPE).numpy()

In [ ]:
# Calculate mean and std
mean_values_NPE = {
    "$\\lvert \\theta - \\theta_* \\rvert$": abs_error_NPE.mean(axis=0),
    "CI90_width": CI90_width_NPE.mean(axis=0),
    "Cover90": Cover90_NPE.mean(axis=0),
    "CI95_width": CI95_width_NPE.mean(axis=0),
    "Cover95": Cover95_NPE.mean(axis=0),
}

std_values_NPE = {
    "$\\lvert \\theta - \\theta_* \\rvert$": abs_error_NPE.std(axis=0),
    "CI90_width": CI90_width_NPE.std(axis=0),
    "CI95_width": CI95_width_NPE.std(axis=0),
}

# Construct the dataframe
df_NPE = pd.DataFrame({
    key: [
        f"{mean:.3f} ({std:.3f})" if key in std_values_NPE else f"{mean}"
        for mean, std in zip(mean_values_NPE[key], std_values_NPE.get(key, [None] * len(mean_values_NPE[key])))
    ]
    for key in mean_values_NPE
})

# Set the index
df_NPE.index = ["Facility"] + [f"Floor {i+1}" for i in range(10)] + ["Room"]

In [ ]:
df_NPE.insert(0, '$\theta_*$', beta_true)

In [ ]:
df_NPE

In [ ]:
df_NPE.to_csv("NPE.csv", index=False)

# Put all results together

In [ ]:
Our_method = pd.read_csv("Our_method.csv")
NPE = pd.read_csv("NPE.csv")
ABCW1 = pd.read_csv("ABCW1.csv")
BSL = pd.read_csv("BSL.csv")

In [ ]:
Our_method.index = ["Facility"] + [f"Floor {i+1}" for i in range(10)] + ["Room"]
NPE.index = ["Facility"] + [f"Floor {i+1}" for i in range(10)] + ["Room"]
ABCW1.index = ["Facility"] + [f"Floor {i+1}" for i in range(10)] + ["Room"]
BSL.index = ["Facility"] + [f"Floor {i+1}" for i in range(10)] + ["Room"]

In [ ]:
# Drop unwanted columns
drop_cols = ["CI90_width", "Cover90"]
Our_method = Our_method.drop(columns=drop_cols)
NPE = NPE.drop(columns=drop_cols)
ABCW1 = ABCW1.drop(columns=drop_cols)
BSL = BSL.drop(columns=drop_cols)

In [ ]:
import pandas as pd

# Step 1: Isolate θ_* column and promote to MultiIndex
theta_star = Our_method[['$\theta_*$']]
theta_star.columns = pd.MultiIndex.from_tuples([('$\\theta_*$', '')])  # <- Fix here

# Step 2: Define metrics (drop θ_* from column list)
metrics = [col for col in Our_method.columns if col != '$\theta_*$']

# Step 3: Build MultiIndex columns (metric, method) format
our_cols = Our_method[metrics]
our_cols.columns = pd.MultiIndex.from_product([metrics, ["Our Method"]])

npe_cols = NPE[metrics]
npe_cols.columns = pd.MultiIndex.from_product([metrics, ["NPE"]])

abcw1_cols = ABCW1[metrics]
abcw1_cols.columns = pd.MultiIndex.from_product([metrics, ["ABCW1"]])

bsl_cols = BSL[metrics]
bsl_cols.columns = pd.MultiIndex.from_product([metrics, ["BSL"]])

# Step 4: Concatenate all method tables horizontally
combined = pd.concat([our_cols, npe_cols, abcw1_cols, bsl_cols], axis=1)
combined = combined.sort_index(axis=1, level=0)  # Optional: sort by metric

# Step 5: Combine with θ_*
res_final = pd.concat([theta_star, combined], axis=1)

# Step 6: Set column level names for nice LaTeX header
res_final.columns.names = ['Metric', 'Method']

In [ ]:
import re

def format_with_makecell(s, metric, method):
    if isinstance(s, str) and re.match(r"^[0-9.]+ \([0-9.]+\)$", s.strip()):
        main, std = s.strip().split(" ")
        return f"\\makecell{{{main} \\\\ {std}}}"
    elif isinstance(s, str):
        return s
    try:
        if metric == '$\\theta_*$' or metric == 'Cover95':
            return f"{s:.2f}"
        else:
            return f"{s:.3f}"
    except:
        return s

for col in res_final.columns:
    metric, method = col
    res_final[col] = res_final[col].apply(lambda x: format_with_makecell(x, metric, method))

In [ ]:
res_final

In [ ]:
# Step 7: Export to LaTeX
latex_code = res_final.to_latex(multicolumn=True, multirow=True, escape=False)

# Output LaTeX
print(latex_code)

# Density plot

In [ ]:
task_id = 0

# weight for ABC and NPE
precond_samples = pd.read_csv(f"res_precond/pre_samples_lam0_task{task_id}.csv")
precond_samples = torch.tensor(precond_samples.values, dtype=torch.float32).contiguous()
mean_theta = precond_samples.mean(dim = 0)
std_theta = precond_samples.std(dim = 0)
dist_prop = torch.distributions.multivariate_normal.MultivariateNormal(loc = mean_theta, covariance_matrix = (std_theta**2).diag())
dist_prior = torch.distributions.multivariate_normal.MultivariateNormal(loc = -3 * torch.ones(12), covariance_matrix = 4 * torch.eye(12))

post_samples = torch.tensor(np.load(f"Langevin_res/post_samples_obs{task_id}.npy"), dtype=torch.float32)

# BSL
log_theta_r1 = torch.tensor(np.load(f"BSL_res/BSL_task_{task_id}.npy"), dtype=torch.float32)
BSL_samples = log_theta_r1.exp()

# ABC_W1
log_theta_r1 = pd.read_csv(f"ABC_res/log_theta_r1_ABCW1_task{task_id}.csv")
log_theta_r1 = torch.tensor(log_theta_r1.values, dtype=torch.float32).contiguous()
ABCW1_samples = log_theta_r1.exp()
# calculate weight: log_weight = log \pi(\theta) - log q(\theta)
log_ABCW1_weight = dist_prior.log_prob(log_theta_r1) - dist_prop.log_prob(log_theta_r1) 
ABCW1_weight = normalized_weight(log_ABCW1_weight)

# NPE
mu = torch.tensor(np.load(f"NPE_res/mu_obs{task_id}.npy"), dtype=torch.float32)
Cov = torch.tensor(np.load(f"NPE_res/Cov_obs{task_id}.npy"), dtype=torch.float32)
log_theta_r1 = torch.distributions.multivariate_normal.MultivariateNormal(
    loc = mu, covariance_matrix = Cov ).sample((10000, ))
NPE_samples = log_theta_r1.exp()
# calculate weight: log_weight = log \pi(\theta) - log q(\theta)
log_NPE_weight = dist_prior.log_prob(log_theta_r1) - dist_prop.log_prob(log_theta_r1) 
NPE_weight = normalized_weight(log_NPE_weight)

In [ ]:
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
    import matplotlib.pyplot as plt
    import seaborn as sns

    fig, axs = plt.subplots(3, 4, figsize=(12, 7))
    axs = axs.flatten()

    for i in range(12):
        sns.kdeplot(x=post_samples[:, i], ax=axs[i], label="n-model", fill=True, alpha=0.1, bw_adjust = 1.1)
        sns.kdeplot(x=ABCW1_samples.cpu()[:, i], weights=ABCW1_weight, ax=axs[i], label="ABC", fill=True, alpha=0.1, bw_adjust = 1.1)
        sns.kdeplot(x=NPE_samples[:, i], weights=NPE_weight, ax=axs[i], label="NPE", fill=True, alpha=0.1, bw_adjust = 1.1)
        sns.kdeplot(x=BSL_samples.cpu()[:, i], ax=axs[i], label="BSL", fill=True, alpha=0.1, bw_adjust = 1.5)

        axs[i].axvline(x=beta_true.cpu()[i], color='black', linestyle='--', label = "Truth")
        axs[i].set_title(f'$\\theta_{{{i}}}$', fontsize = 16)
        axs[i].legend()  # make sure a legend exists (add this line!)
        axs[i].set_xlabel("")
        axs[i].set_ylabel("")

        if i == 1:
            axs[i].set_xlim(0, 0.4)

        if i == 2:
            axs[i].set_xlim(0, 0.4)

        if i == 3:
            axs[i].set_xlim(0, 0.5)

        if i == 4:
            axs[i].set_xlim(0, 0.5)

        if i == 11:
            axs[i].set_xlim(0, 0.5)

    # Collect handles and labels from the first axis
    axs[0].set_ylabel("Density", fontsize=14)
    axs[4].set_ylabel("Density", fontsize=14)
    axs[8].set_ylabel("Density", fontsize=14)
    handles, labels = axs[0].get_legend_handles_labels()
    
    # Remove individual legends safely
    for ax in axs:
        ax.tick_params(axis='both', labelsize=14)
        if ax.get_legend() is not None:
            ax.get_legend().remove()

    # Add a single legend outside the plot
    fig.legend(handles, labels, loc='lower center', ncol=5, bbox_to_anchor=(0.5, -0.1), fontsize = 16)

    plt.tight_layout(rect=[0, 0.0, 1, 0.95]) 
    plt.show()

In [ ]:
# fig.savefig("Figures/represent_SI_10FSS_kde.pdf", dpi=300, bbox_inches='tight')

In [ ]:
data_obs = pd.read_csv(f"data_obs/y_obs_task{task_id}.csv")
data_obs = torch.tensor(data_obs.values, dtype=torch.float32).contiguous().to(device)

plt.figure(figsize=(6, 5))
plt.plot(data_obs.cpu().sum(dim = 0) / N, label = "Total", color = "black")
for k in range(K):
    plt.plot(data_obs.cpu()[F_assign.cpu()[:, k] == 1, :].sum(dim = 0) / (F_assign.cpu()[:, k] == 1).sum(), label = f"Floor {k+1}")
plt.legend()
# plt.ylim(0, 1)
plt.title("Observed data, ratio of infection")
# plt.savefig("Figures/represent_SI_10FSS_dataobs.pdf", dpi=300, bbox_inches='tight')
plt.show()